## **How can I use this method: LIME in practice**

Если разбираться в деталях, то под капотом LIME также делает (или может делать) интересные преобразования, которые способны усилить и специализировать объяснение. Также, благодаря взгляду на использование метода с разных сторон, можно получить несколько различных вариантов объяснения, используя гиперпараметры методов библиотеки.

Вместо того, чтобы использовать оболочку LIME в этом практическом занятии мы разберемся со всем и построим две собственные суррогатные модели for scratch И воссоздадим подход из [этой](https://arxiv.org/pdf/1910.13016) статьи. Прохождение этого ноутбука поможет вам "чувствовать" алгоритм и использовать его с умом.  

Приступим!

[![temp-Image1mlt-QZ.avif](https://i.postimg.cc/LXny3rrP/temp-Image1mlt-QZ.avif)](https://postimg.cc/hzRbS344)



In [ ]:
!pip install lime fat-forensics[all] -q #установка необходимых библиотек

In [ ]:
import fatf
import lime
import pandas as pd
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

fatf.setup_random_seed(42)

Подготовим данные. Для простоты, будем работать с очень известным в статистике датасетом — Ирисы Фишера. Он представляет собой измерения, снятые со 150 экземпляров ириса, по 50 экземпляров каждого из трёх видов — Ирис щетинистый (Iris setosa), Ирис виргинский (Iris virginica) и Ирис разноцветный (Iris versicolor).

Для них представлены по четыре характеристики (в сантиметрах):

- Длина наружной доли околоцветника (англ. sepal length);
- Ширина наружной доли околоцветника (англ. sepal width);
- Длина внутренней доли околоцветника (англ. petal length);
- Ширина внутренней доли околоцветника (англ. petal width).

На основании этого набора данных требуется построить правило классификации, определяющее вид растения по данным измерений. Это задача многоклассовой классификации, так как имеется три класса — три вида ириса.

In [ ]:
#Загрузка

iris_data_dict = load_iris()
iris_data = iris_data_dict['data']
iris_target = iris_data_dict['target']
iris_feature_names = iris_data_dict['feature_names']
iris_target_names = iris_data_dict['target_names']

X_train, X_test, y_train, y_test = train_test_split(iris_data, iris_target, random_state=42)

В качестве модели "черного ящика" будем исползовать случайный лес.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import sklearn.metrics

blackbox_model = RandomForestClassifier(random_state=42, max_depth=2)
blackbox_model.fit(X_train, y_train)

predictions = blackbox_model.predict(X_test)
acc = sklearn.metrics.accuracy_score(y_test, predictions)

print(f'Model accuracy: {acc}')

In [ ]:
data_point = X_train[37] #Выберем произвольную точку данных

data_point_probabilities = blackbox_model.predict_proba(data_point.reshape(1, -1))[0]
data_point_probabilities

In [ ]:
data_point

In [ ]:
data_point_prediction = data_point_probabilities.argmax(axis=0) #Смотрим на класс для точки данных

data_point_class = iris_target_names[data_point_prediction]
data_point_class

Отлично! Рассмотрим разноцветный ирис. Чтобы в прямом смысле посмотреть на него, визуализируем набор данных и выбранную точку данных.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


_ = plt.figure()
_ = plt.scatter(
     X_train[y_train==0][:, 2],
     X_train[y_train==0][:, 3],
     label=iris_target_names[0])
_ = plt.scatter(
     X_train[y_train==1][:, 2],
     X_train[y_train==1][:, 3],
     label=iris_target_names[1])
_ = plt.scatter(
     X_train[y_train==2][:, 2],
     X_train[y_train==2][:, 3],
     label=iris_target_names[2])
_ = plt.scatter(
     data_point[2],
     data_point[3],
     label='Explained Data Point',
    s=100, c='k')

_ = plt.xlabel(iris_feature_names[2])
_ = plt.ylabel(iris_feature_names[3])
_ = plt.legend()

_ = plt.title('Наблюдаемые и интерпретируемое значение')

## Build LIME model

Вооружимся некоторыми теоретическими сведениями об алгоритме. При обучении суррогатной модели $g(z)$ для модели $f(x)$ мы решаем задачу минимизации вида:

$$\xi(x) = argmin_{g \in G} L(f, g, \pi_x) + \Omega(g),$$

где
- $G$ — множество интерпретируемых моделей, $g \in G$ некоторая интерпретируемая модель (линейная регрессия, дерево решений и др.)
- $f(x)$ — модель — черный ящик (та, которую мы хотим объяснить)
- $\Omega(g)$ — параметр, отвечающий на сложность суррогатной модели.


Здесь важно обратить внимание на то что в в функции потерь $L$ присутствует нестандартный параметр $\pi_x$. Он отвечает за корректировку вклада объектов из окресности, назначая им веса и в функции $L$ является некоторой поправкой при вычислении квадратичной ошибки.

$$L(f, g, \pi_x) = \pi_x(f(z) - g(x'))^2,$$

$\pi(x)$ для каждого $x$ рассчитывается согласно формуле $exp(\frac{-D(x', x)}{\sigma^2})$  (под $D(. , .)$ подразумевается расстояние, $\sigma$ — ширина ядра окрестности).

В оригинальной имплементации ([“Why Should I Trust You?” Explaining the Predictions of Any Classifier](https://arxiv.org/pdf/1602.04938)) обучать локальную суррогатную модель можно с *дискретизацией* непрерывных признаков и без, а для некоторых модальностей данных (текст, изображения) используется бинаризация. Эти понятия могут быть незнакомы, однако на практике они позволяют взглянуть на построенные суррогатные модели с разных сторон и получить разные интерпретации признаков.


In [ ]:
import fatf.utils.data.discretisation as fatf_discretisation #импортируем вспомогательные функции
import fatf.utils.data.augmentation as fatf_augmentation

**Шаг 1.** Первый шаг алгоритма LIME генерация данных, похожих на исходные.

In [ ]:
augmenter = fatf_augmentation.Mixup(X_train, ground_truth=y_train)

sampled_data = augmenter.sample(data_point, samples_number=100) #Сгенерировали точки на основе исходных
sampled_data_probabilities = blackbox_model.predict_proba(sampled_data) #Спрогнозировали вероятности для сгенерированных точек

Посмотрим на новые точки.

In [ ]:
sampled_data_predictions = sampled_data_probabilities.argmax(axis=1)
sampled_data_0_indices = np.where(sampled_data_predictions == 0)[0]
sampled_data_1_indices = np.where(sampled_data_predictions == 1)[0]
sampled_data_2_indices = np.where(sampled_data_predictions == 2)[0]

_ = plt.figure()
_ = plt.scatter(
     X_train[y_train==0][:, 2],
     X_train[y_train==0][:, 3],
     label=iris_target_names[0])
_ = plt.scatter(
     X_train[y_train==1][:, 2],
     X_train[y_train==1][:, 3],
     label=iris_target_names[1])
_ = plt.scatter(
     X_train[y_train==2][:, 2],
     X_train[y_train==2][:, 3],
     label=iris_target_names[2])
_ = plt.scatter(
     data_point[2],
     data_point[3],
     label='Explained Data Point',
    s=100, c='k')


_ = plt.scatter(
     sampled_data[sampled_data_0_indices, 2],
     sampled_data[sampled_data_0_indices, 3],
     label='Augmented Data: {}'.format(iris_target_names[0]))
_ = plt.scatter(
     sampled_data[sampled_data_1_indices, 2],
     sampled_data[sampled_data_1_indices, 3],
     label='Augmented Data: {}'.format(iris_target_names[1]))
_ = plt.scatter(
     sampled_data[sampled_data_2_indices, 2],
     sampled_data[sampled_data_2_indices, 3],
     label='Augmented Data: {}'.format(iris_target_names[2]))

_ = plt.xlabel(iris_feature_names[2])
_ = plt.ylabel(iris_feature_names[3])
_ = plt.legend()

**Наблюдения:**

Отметим, что синтетические точки **похожи** на исходные и находятся близко **ко всем** классам. Однако это не единственный подход к созданию синтетических данных для прогонза. Новые данные также можно генерировать **только в окресности** объясняемой точки. За это в официальной реализации Lime отвечает гиперпараметр `sample_around_instance`.


Мы же реализуем подход, когда генерация синтетических точек делается на основе **всех** данных.

```
**Запомнить:** первое, что можно поменять в LIME, чтобы получить иное объяснение - окресность точки.
```

**Шаг 2. Дискретизация и бинаризация данных.**

**Дискретизация** — процесс преобразования некоторой непрерывной функции к диксретной. В данном случае (и это является *одной из возможных *реализаций алгоритма LIME из коробки), дискретизация происходит при помощи разбиения на *квартили*.

**Квартили** — это значения, которые делят упорядоченный набор данных на четыре равных части, каждая из которых содержит 25% данных. Процесс дискретизации (разбиения) на квартили называется квартильной дискретизацией.
Она происходит следующим образом:
1. Извлекаем границы каждого квартиля
2. Каждую точку в наборе данных кодируем как вектор с координатами из мноества {0, 1, 2, 3} по правилу — заменяем значение координаты на номер квартиля (0, 1, 2, 3), в который она попадает.

**Quize:** Какие координаты у вектора $[2.3, 5.1, 2., 3.3]$ получатся, если границы квартилей такие: $[0, 2.1], [2.2, 3], [3.1, 4], [4.1, 5.]$

В ответ запишите координату номер 2 (нумерация координат с единицы).

In [ ]:
discretiser = fatf_discretisation.QuartileDiscretiser(
    X_train,
    feature_names=iris_feature_names)

data_point_discretised = discretiser.discretise(data_point)
sampled_data_discretised = discretiser.discretise(sampled_data)

discretiser.feature_bin_boundaries

Убедимся, что значения границ — действительно квартили.

In [ ]:
import pandas as pd

pd.DataFrame(X_train, columns=iris_feature_names).describe()[3:] #Дествительно, границы совпадают с нужными нам статистиками.

In [ ]:
#Посмотрим на дискретизированную точку
data_point_discretised

Теперь, чтобы получить *интерпретируемую модель* осуществим **бинаризацию данных**.
Бинаризация данных — как следут из названия — приведение всех координат к сочетанию нулей и единиц (бинарному виду). Бинаризовать будем по следующему правилу: ставить 0, если координаты дискретизированной точки данных не совпадают с точкой данных из сэмпла, и 1 иначе.

In [ ]:
import fatf.utils.data.transformation as fatf_transformation

sampled_data_binarised = fatf_transformation.dataset_row_masking(
    sampled_data_discretised, data_point_discretised)

fatf_transformation.dataset_row_masking(data_point_discretised.reshape(1, -1), data_point_discretised)

На практике бинаризация осуществляется для текстовых данных и изображений. На нашем примере она даст результаты, которые будут рассогласованы с другими подходами к интерпретации. Какие лучше — необходимо проверять на практике.

```
**Запомнить:** дискретизация и бинаризация также позволяют более разносторонне использовать один алгоритм объяснения.
```

Ещё раз рассмотрим все 3 трансформации:

In [ ]:
print(f'Исходная точка данных: {data_point}')
print(f'Дискретизированная точка данных: {data_point_discretised}')

print(f'Некоторый экземпляр из дискретизированных точек до бинаризации: {sampled_data_discretised[7]}')
print(f'Некоторый экземпляр из дискретизированных точек после бинаризации: {sampled_data_binarised[7]}')

В качестве последнего шага осталость вычислить веса объектов для функции ошибки. Это  мы просто сделаем по соответствующей формуле, которую вы вспоминали на задаче выше.

In [ ]:
import fatf.utils.distances as fatf_distances
import fatf.utils.kernels as fatf_kernels

features_number = sampled_data_binarised.shape[1]
kernel_width = np.sqrt(features_number) * 0.75

distances = fatf_distances.euclidean_point_distance(np.ones(features_number), sampled_data_binarised) #расстояния вычислим на основе бинаризованных данных
weights = fatf_kernels.exponential_kernel(
     distances, width=kernel_width)

## **Обучение локального алгоритма**

Задача, решаемая моделями для датасета с ирисами Фишера, является задачей мультиклассовой классификации. Таким образом, тренируя суррогатную модель мы можем обучить её двумя способами:
- используя подход ONE VS REST, где суррогатная модель будет прогнозировать 0 или 1 в зависимости от факта принадлежности объекта к интересующему нас классу.
- используя классический подход, где суррогатная модель будет прогнозировать вектор вероятностей.

Преимущество первого подхода — концентрация на интересующем нас классе, второго — на универсальности — полученного суррогата можно использовать для объяснения объекта из любых классов.

Кроме того, выше мы отметили, что модель может быть обучена на бинаризованных данных и нет. Обучая на бинаризованных данных, получается модель, отвечающая на вопрос:

*«Если бы это конкретное значение признака объясняемой точки данных находилось за пределами диапазона (для числовых признаков) или имело другое значение (для категориального признака), как бы это повлияло на вероятность принадлежности этой точки к объясняемому классу (вероятностная классификация) / прогнозируемое числовое значение (регрессия)?»*

Это полезно для изображений, так как позволяет сранивать знаимость различных сегментов картинки.

В нашем же случае модель решим задачу минимизации прогнозируемых значений от истинных в классическом смысле.

```
**Запомнить:** расстояние точек от объясняемого объекта также позволяет осуществлять тюнинг объяснения.
```

In [ ]:
sampled_data_predictions_versicolor = sampled_data_probabilities[:, 1] # сохраним вероятности для подхода OVR

Инициализируем модель и обучим её на дискретизированных данных на всех вероятностях.

In [ ]:
import sklearn.linear_model
import numpy as np

In [ ]:
lime_model = sklearn.linear_model.Ridge(alpha=1, fit_intercept=True)

lime_model.fit(sampled_data_discretised, sampled_data_predictions, sample_weight=weights)
for name, importance in zip(iris_feature_names, lime_model.coef_):
     print('->{}<-: {}'.format(name, importance))

**И вот вы обучили первую LIME!**
Зафиксируем важные признаки по порядку: `petal width`, `petal lenght`

Посмотрим, на реализованный вариант на не дискретизированных данных

In [ ]:
lime_model = sklearn.linear_model.Ridge(alpha=1, fit_intercept=True)

lime_model.fit(sampled_data, sampled_data_predictions, sample_weight=weights)  #посмотрим, на реализованный вариант на не дискретизированных данных

for name, importance in zip(iris_feature_names, lime_model.coef_):
     print('->{}<-: {}'.format(name, importance))

Зафиксируем важные признаки по порядку здесь: petal width, sepal lenght

**Quize: Поменяйте подход вычисления расстояний — вычислите их на основе оригинальных значений данных. Обучите локальную модель на дискретизированных данных. Поменялись ли 2 самых важных признака?**

In [ ]:
distances2 = fatf_distances.euclidean_point_distance(data_point, sampled_data) #расстояния вычислим на основе оригинальных данных
weights2 = fatf_kernels.exponential_kernel(
     distances2, width=kernel_width)

lime_model = sklearn.linear_model.Ridge(alpha=1, fit_intercept=True)

lime_model.fit(sampled_data_discretised, sampled_data_predictions, sample_weight=weights2)  #посмотрим, на реализованный вариант на не дискретизированных данных

for name, importance in zip(iris_feature_names, lime_model.coef_):
     print('->{}<-: {}'.format(name, importance))

Ответ:

**Некоторые факты**
- Также топ-2 признака не поменяются, если расстояния вычислить на основе дискретизированных данных.
-  Также топ-2 признака не поменяются, если строить модель на оригинальных данных.

**Библиотечная реализация**. \
Посмотрим, что дает библиотечная реализация. Все гиперпараметрв достаточно интуитивны, но на всякий случай поясним каждый:

- `class_names` — названия прогнозируемых классов
- `feature_names` — названия принаков
- `kernel_width`— ширина ядра, по умолчанию (и в нашем случае) $\sqrt{n\_features}*0.75$
- `verbose` — подробные выкладки при обучении суррогатной модели
- `discretizer` — подход к дискретизации
- `mode` — задача, реашемая моделью
- `discretize_continuous` — необходимо ли дискретизировать признаки
- `sample_around_instance` — генерировать ли синтетические данные **только в окресности** рассматриваемой точки

Последовательно посмотрим на важные признаки на дискретизированных и нет данных. Начнем с недискритиpированных.

```
**Запомнить:** ширина ядра окресности также позволяет осуществлять тюнинг объяснения, однако наиболее часто используется ширина
по умолчанию.
```


In [ ]:
from lime.lime_tabular import LimeTabularExplainer #посмотрим, что дает библиотечная реализация

explainer = LimeTabularExplainer(X_train,
                                 class_names=iris_target_names,
                                 feature_names=iris_feature_names,
                                 kernel_width=np.sqrt(features_number) * 0.75,
                                 verbose=False,
                                 mode='classification',
                                 discretize_continuous=False,
                                 sample_around_instance=False)

exp = explainer.explain_instance(data_point, blackbox_model.predict_proba, top_labels=1)
exp.show_in_notebook()

Видим, что без дискретизации не видно явной силы влияния конкретных признаков.

**Quize: Получите интерпретацию с квартильной дискретизацией. Согласуется ли она с полученной выше при ручной реализации?**

In [ ]:
from lime.lime_tabular import LimeTabularExplainer #посмотрим, что дает библиотечная реализация

explainer = LimeTabularExplainer(X_train,
                                 class_names=iris_target_names,
                                 feature_names=iris_feature_names,
                                 kernel_width=np.sqrt(features_number) * 0.75,
                                 verbose=False,
                                 discretizer='quartile',
                                 mode='classification',
                                 discretize_continuous=True,
                                 sample_around_instance=False)

exp = explainer.explain_instance(data_point, blackbox_model.predict_proba, top_labels=1)
exp.show_in_notebook()

Видим, что результаты не равны друг другу в точноти, но похожи по общим выводам.

**Quize: Получите интерпретацию при помощи ручной реализации на бинаризованных данных на весах `weights`. Какие признаки (признак) подсвечиваются в этом случае?**

In [ ]:
distances2 = fatf_distances.euclidean_point_distance(data_point, sampled_data) #расстояния вычислим на основе оригинальных данных
weights2 = fatf_kernels.exponential_kernel(
     distances2, width=kernel_width)

lime_model = sklearn.linear_model.Ridge(alpha=1, fit_intercept=True)

lime_model.fit(sampled_data_binarised, sampled_data_predictions, sample_weight=weights)  #посмотрим, на реализованный вариант на не дискретизированных данных

for name, importance in zip(iris_feature_names, lime_model.coef_):
     print('->{}<-: {}'.format(name, importance))

**Ответ: `petal width (cm)` и `petal length (cm)`**

## **Суррогатное дерево**

Линейная модель — не единственная в нашем арсенале. В некоторых случаях более полезным и информативным оказывается построение суррогатного дерева. Простой пример построения:

In [ ]:
import sklearn.tree

blimey_tree = sklearn.tree.DecisionTreeClassifier(max_depth=3, random_state=42)
blimey_tree.fit(sampled_data, sampled_data_predictions, sample_weight=weights)


In [ ]:
for n_i in zip(iris_feature_names, blimey_tree.feature_importances_):
     name, importance = n_i
     print('->{}<-: {}'.format(name, importance))


Также для интерпретации здесь полезно визуализировать правила разбиений.

In [ ]:
from sklearn import tree
print(tree.export_text(blimey_tree))

Или более красиво.

In [ ]:
import graphviz

dot_data = tree.export_graphviz(blimey_tree, out_file=None,
                                feature_names=iris_feature_names,
                                class_names=iris_target_names,
                                filled=True)


graph = graphviz.Source(dot_data, format="png")
graph

**Quize: Какой признак отсутствует в структуре построенного суррогатного дерева?** \
Ответ:

### **Выводы**
- LIME допускает разные подходы к формированию и оценке важности признаков
- Объяснения внутри различных сценариев LIME могут различаться, так что истинность необходимо проверять эмпирически, а гипотезы генерировать на основе комбинации методов/моделей
- Использование LIME предполагает, что используются изначально концептуально понятные признаки (кусочки изображения, значения числового вектора, соотносящиеся с реальными данными)
- Благодаря гибкости гиперпараметров алгоритма, вы можете анализировать не только одно объяснение, но и его устойчивость в зависимости от данных.

### **Hints to tuning LIME:**
1. Первое, что можно поменять в LIME, чтобы получить иное объяснение - окресность точки или ширину ядра.
2. Второй подход к тюнингу — способы для рассчета значимости расстояния объектов до точки.
3. Ширина ядра окресности также позволяет осуществлять тюнинг объяснения, однако наиболее часто используется ширина по умолчанию.
4. Дикретизация и бинаризация также позволяют более разносторонне использовать один алгоритм объяснения, получая ответы на другие формулировки вопроса к объекту.

Спасибо за ваше время на эту практику! Надеюсь, она поможет сделать ваши модели более интерпретируемыми.

До встречи! Ваш,\
Дата Автор =)

telegram: https://t.me/sabrina_sadiekh \
[LinkedIn](https://www.linkedin.com/in/sabrina-sadiekh-35181a286/) \
My course about explainable AI: https://open-xai-platform.web.app \